# Step 5 · Error analysis & judges

Label answers, count failure modes, measure reviewer agreement, validate a judge, gate a release.

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
SAVE = False   # set True to write new versions into your lab (they appear in the dashboard)

In [ ]:
from lab import evals, judge
e = registry.evals()[0] if registry.evals() else None
assert e, "Run notebook 04 (or Evaluation & Scoring) first"
R = e["results"]; print(f"evaluation #{e['id']} · {len(R)} answers")

In [ ]:
# Label answers (here: a toy rule so the cell runs — in real life you read each answer)
for x in R:
    if not x.get("review"):
        x["review"] = "correct" if "check" in x["answer"].lower() else "incorrect"
        x["failure_mode"] = "" if x["review"] == "correct" else "no check before promising"
pd.DataFrame(evals.failure_table(R), columns=["failure mode", "count", "share"])

In [ ]:
# Agreement between two reviewers (Cohen's kappa corrects for chance)
me    = [x["review"] for x in R]
other = [("incorrect" if i % 4 == 0 else lab) for i, lab in enumerate(me)]
k, n = evals.cohen_kappa(me, other); print(f"kappa {k:.2f} ({evals.kappa_meaning(k)}) on {n} answers")

In [ ]:
# Can the rule score be trusted as a judge? TPR/TNR + bias-corrected pass rate with a bootstrap CI
rep = evals.judge_report(R, threshold=0.5)
{k: rep[k] for k in ("tp", "fn", "tn", "fp", "tpr", "tnr", "judge_pass_rate", "corrected_pass_rate", "ci95")}

In [ ]:
# LLM-as-judge on one answer (validate it against your labels before trusting it)
x = R[0]
judge.grade(BASE, x["question"], x["answer"], x.get("approved", ""))

In [ ]:
ok, msgs = evals.gate(R, e["mean_reward"], min_score=0.5, min_human=0.7)
print("GATE", "PASSED" if ok else "FAILED", msgs)
if SAVE:
    import json; registry.update("evals", e["id"], results_json=json.dumps(R))